# core

> `astream` and `ClaudeRun`: stateless user turns with in-place caller-owned tool continuation through Claude Code

In [ ]:
#| default_exp core

`astream(msgs, ...)` returns a `ClaudeRun` through the installed `claude`, using its login and subscription. Each top-level user turn starts from the complete canonical history, compiled into a native transcript under a fresh session id. Within that turn, caller-owned tools use Claude's ordinary MCP flow: iteration stops after the complete streamed tool batch, `resume()` supplies its `ToolResult`s, and the same process continues. Iteration yields raw stream-json events; `.messages` accumulates the canonical trace (signatures intact, tool names unqualified), `.paused` distinguishes a tool boundary, and `.result` holds the terminal result. `run.interrupt()` ends a turn natively, and closing interrupts, drains, escalates to terminate/kill, and removes the transcript. Runs default to an isolated XDG cache work dir; pass `cwd=` for project context and `native_tools=` for built-ins such as `WebSearch`.


In [ ]:
#| export
import asyncio, json, os, shutil, uuid
from contextlib import suppress
from fastcore.utils import *
from fastcore.meta import delegates
from fastcore.xdg import xdg_cache_home
from fastllm.anthropic import denorm_msgs, norm_parts
from aidialog.msg_parts import Msg, Text, ToolUse, ToolResult, Media
from fastclaude.session import *
from fastclaude.protocol import *

In [ ]:
from fastcore.test import *
import stat, sys, tempfile, textwrap

## The work dir

In Claude Code, the working directory is a storage key: the transcript we fabricate must sit in the `~/.claude/projects` folder derived from the directory the process runs in, or `--resume` finds nothing. It is also a behavior input, deciding which project settings and CLAUDE.md join the run, and whose session list our synthetic transcripts would pollute. So by default each run gets an isolated pseudo-project under the XDG cache dir: answers stop depending on where the host process happens to sit, no real project's sessions are touched, and everything filed under the cache path's project folder is ours to delete. Passing `cwd=` opts in to a real project's context instead.

In [ ]:
#| export
MCP_SERVER = 'fastclaude'
MCP_PREFIX = f'mcp__{MCP_SERVER}__'
SERVER_TOOLS = ('WebSearch','WebFetch')

def work_dir():
    "The default run directory: an isolated pseudo-project under the XDG cache dir"
    p = xdg_cache_home()/'fastclaude'
    p.mkdir(parents=True, exist_ok=True)
    return p

## Compiling history

Every call supplies the complete history as `aidialog.msg_parts.Msg` objects ending in a genuine user prompt (text or media). That prompt becomes the live turn and everything before it becomes the native transcript. Tool results no longer start another process: they resume the already-paused `ClaudeRun`. `denorm_msgs` converts the history to Anthropic-style wire messages, and `prefix_tools` qualifies callable-tool names the way this run offers them, leaving Claude Code's own tool names alone.


In [ ]:
#| export
def compile_msgs(
    msgs, # Complete history as `Msg`s, ending with a user prompt
):
    "`(history, prompt)`: wire messages to file and the live turn's content"
    msgs = listify(msgs)
    if not msgs: raise ValueError('empty message history')
    den = prefix_tools(denorm_msgs(msgs), MCP_PREFIX, skip=SERVER_TOOLS)
    content = den[-1]['content']
    if msgs[-1].role=='user' and any(isinstance(p, (Text,Media)) for p in msgs[-1].content): return den[:-1],content
    raise ValueError('history must end with a user prompt')

In [ ]:
prior = Msg('user', [Text('Measure the flux please.')])
call = Msg('assistant', [Text('Checking.'), ToolUse(id='t1', name='flux_meter', arguments={})])
result = Msg('tool', [ToolResult(id='t1', name='flux_meter', text='flux: 41.7 kf')])
prompt = Msg('user', [Text('And in gauss?')])
h = [prior,call,result,prompt]
h

`compile_msgs` files the first three messages as native history, qualifying its external tool name, and returns only the final user's content for the live turn.

In [ ]:
hist,prompt = compile_msgs(h)
test_eq(len(hist), 3)
test_eq(hist[1]['content'][1]['name'], 'mcp__fastclaude__flux_meter')
test_eq(prompt, [dict(type='text', text='And in gauss?')])
hist[1]

A run needs a genuine new user turn. Empty input and history ending at an earlier tool result are both rejected before any process starts.

In [ ]:
with expect_fail(ValueError, contains='user prompt'): compile_msgs(h[:3])
with expect_fail(ValueError, contains='empty'): compile_msgs([])

## The command and environment

The executable is the user's installed `claude`, with their real config and login: that is the whole point, and it is why `CLAUDE_CONFIG_DIR` is never isolated. `--strict-mcp-config` is always passed, keeping their other configured MCP servers out of every run (without it, a run with no tools of its own will happily use whatever MCP servers the user has configured), `--tools` with an explicit list (empty by default) disables built-in tools until asked for, and the private SDK server entry in `--mcp-config` is how our in-process tools join. `--resume` uses the equals form so a dash-leading session id can never parse as a flag of its own. `setting_sources` picks which of the user's settings join the run: None keeps the CLI default (all of them), `['project']` loads only the project's, and `()` loads none. `mcp_config` adds external MCP servers, such as a stdio process, beside the private SDK entry. The environment blanks `ANTHROPIC_API_KEY`, so an inherited key cannot silently bill the API, and removes `CLAUDECODE`, so the child does not believe it is nested inside another Claude Code.

In [ ]:
#| export
def claude_cmd(
    model=None, # Model alias or full name; None uses the user's default
    resume=None, # Session id to resume, i.e. the transcript just written
    system=None, # System prompt; None keeps Claude Code's own
    tools=False, # Offer the private SDK MCP server?
    native_tools=(), # Built-in Claude Code tools to enable, e.g. 'WebSearch'
    allowed=(), # `--allowedTools` entries, e.g. qualified callable names
    append_system=None, # Text appended to Claude Code's own system prompt, which stays
    setting_sources=None, # Settings that load, e.g. ['project']; () loads none; None keeps the CLI default (all)
    mcp_config=None, # Extra MCP server entries, e.g. `dict(clikernel=dict(type='stdio', command=...))`
    max_turns=None, # Bound on agent turns; None is unbounded
    max_budget=None, # Max USD for the run; None is unbounded
    permission_mode=None, # e.g. 'bypassPermissions'; None keeps the CLI default
    thinking=None, # 'adaptive' or 'disabled'; None keeps the CLI default
    effort=None, # Thinking effort: 'low', 'medium', or 'high'
    claude_path=None, # Explicit claude executable; found on PATH if None
):
    "argv for one headless stream-json claude run"
    c = [str(claude_path or shutil.which('claude') or 'claude'), '--output-format','stream-json',
        '--input-format','stream-json', '--verbose', '--include-partial-messages']
    if model: c += ['--model', model]
    if system is not None: c += ['--system-prompt', system]
    if append_system: c += ['--append-system-prompt', append_system]
    if resume: c += [f'--resume={resume}']
    servers = dict(mcp_config or {})
    if tools: servers[MCP_SERVER] = dict(type='sdk', name=MCP_SERVER)
    if servers: c += ['--mcp-config', json.dumps(dict(mcpServers=servers))]
    c.append('--strict-mcp-config')
    c += ['--tools', ','.join(native_tools)]
    if allowed: c += ['--allowedTools', ','.join(allowed)]
    if setting_sources is not None: c.append(f"--setting-sources={','.join(setting_sources)}")
    for f,v in (('--max-turns',max_turns), ('--max-budget-usd',max_budget), ('--permission-mode',permission_mode), ('--thinking',thinking), ('--effort',effort)):
        if v is not None: c += [f, str(v)]
    return c

def claude_env():
    "A child environment that cannot bill an API key and does not think it is nested"
    env = dict(os.environ, ANTHROPIC_API_KEY='')
    env.pop('CLAUDECODE', None)
    return env

In [ ]:
c = claude_cmd('sonnet', resume='-abc', tools=True, allowed=['mcp__fastclaude__flux_meter'])
test('--resume=-abc', c, in_)
test_eq(c[c.index('--tools')+1], '')
test('--strict-mcp-config', c, in_)
test('--strict-mcp-config', claude_cmd('sonnet'), in_)
test_eq(json.loads(c[c.index('--mcp-config')+1]), dict(mcpServers=dict(fastclaude=dict(type='sdk', name='fastclaude'))))
env = claude_env()
test_eq(env['ANTHROPIC_API_KEY'], '')
assert 'CLAUDECODE' not in env
c[1:]

The run-shaping options render only when given, so the default command stays minimal:

In [ ]:
c2 = claude_cmd(mcp_config=dict(clikernel=dict(type='stdio', command='clikernel-mcp')), setting_sources=['project'], max_turns=8)
test_eq(json.loads(c2[c2.index('--mcp-config')+1])['mcpServers']['clikernel']['type'], 'stdio')
test('--setting-sources=project', c2, in_)
test_eq(c2[c2.index('--max-turns')+1], '8')
assert '--max-turns' not in claude_cmd()
c2[1:]

## The run

A `ClaudeRun` owns one temporary transcript and one process. Its first iteration compiles and files the history, spawns Claude, performs the handshake, sends the live prompt, and yields every non-control event raw. Full assistant events become canonical `Msg`s, tool-result user events become tool-role messages, and terminal output lands on `.result`. When an advertised tool batch reaches `message_stop`, iteration instead returns with `.paused=True`; the process and event iterator remain alive for `resume()` and the next iteration.


In [ ]:
#| export
class ClaudeRun:
    "One Claude process whose caller-owned tool rounds pause and resume in place"
    @delegates(claude_cmd, but=['model','resume','tools','native_tools','allowed'])
    def __init__(self,
        msgs, # Complete history as `Msg`s, ending with a user prompt
        model='sonnet', # Model alias or full name
        tools=None, # Tool schemas to advertise, in either `tool_spec` form; the caller executes
        cwd=None, # Project directory for the run; the isolated `work_dir()` if None
        native_tools=(), # Built-in Claude Code tools to enable, e.g. 'WebSearch'
        allowed=(), # Extra `--allowedTools` entries beyond the advertised schemas
        env=None, # Extra child environment variables, merged over `claude_env()`
        **kwargs, # Passed to `claude_cmd`, e.g. `system`, `setting_sources`, `mcp_config`
    ):
        store_attr('msgs,model,tools,cwd,native_tools,allowed,env')
        self.cmd_kwargs = kwargs
        self.messages,self.result,self.proc,self.proto = [],None,None,None
        self.paused = False
        self._names,self._batch = {},[]
        self._pause_ids,self._closed,self._spath,self._events = [],False,None,None

    def __aiter__(self): return self._turn()

@delegates(ClaudeRun)
def astream(msgs, **kwargs):
    "Start one completion; each iteration ends at a caller-owned tool batch or the terminal result"
    return ClaudeRun(msgs, **kwargs)

Spawning files the history and builds the one process used by every tool round in this user turn. A fresh random session id avoids collisions when identical histories run concurrently; prompt caching depends on request content, not session reuse. An empty history (a bare first prompt) writes no transcript and resumes nothing:


In [ ]:
#| export
@patch
async def _spawn(self:ClaudeRun):
    "Compile and file the history, then start Claude and return the live prompt"
    self.cwd = Path(self.cwd).expanduser() if self.cwd else work_dir()
    hist,prompt = compile_msgs(self.msgs)
    sid = str(uuid.uuid4())
    recs = msgs2recs(hist, key=sid, cwd=self.cwd)
    if recs:
        save_sess(recs, sid, self.cwd)
        self._spath = sess_file(sid, self.cwd)
    schemas = mk_tools(self.tools or [])
    allowed = [MCP_PREFIX+s['name'] for s in schemas] + list(self.native_tools) + list(self.allowed)
    argv = claude_cmd(self.model, resume=sid if recs else None, tools=bool(schemas),
        native_tools=self.native_tools, allowed=allowed, **self.cmd_kwargs)
    self.proc = await asyncio.create_subprocess_exec(*argv, stdin=asyncio.subprocess.PIPE,
        stdout=asyncio.subprocess.PIPE, limit=2**25, cwd=self.cwd, env=dict(claude_env(), **(self.env or {})))
    self.proto = ClaudeProto(self.proc, tools=self.tools, server=MCP_SERVER)
    self._events = self.proto.events().__aiter__()
    return prompt

Trace accumulation reads only full message events, ignoring partials: they are transcript-grade, one event per content block. Callable names are unqualified on the way back, so callers see the tool they registered rather than Claude's `mcp__` spelling. Those external ToolUses also form the current batch. Results later emitted by the same CLI process are folded into the trace normally, along with native tool results such as `WebSearch`:


In [ ]:
#| export
def unqual(nm):
    "The bare tool name for a possibly `mcp__fastclaude__`-qualified `nm`"
    return nm[len(MCP_PREFIX):] if nm and nm.startswith(MCP_PREFIX) else nm

def _flat(c): return c if isinstance(c, str) else '\n'.join(b.get('text','') for b in c if b.get('type')=='text')

@patch
def _track(self:ClaudeRun, m):
    "Fold one raw event into `.messages`, the current external tool batch, and `.result`"
    t,c = m.get('type'), nested_idx(m, 'message', 'content')
    if t=='assistant' and isinstance(c, list):
        parts = norm_parts(m['message'])
        for p in parts:
            if isinstance(p, ToolUse):
                external = p.name.startswith(MCP_PREFIX)
                p.name = self._names[p.id] = unqual(p.name)
                if external: self._batch.append(p)
        self.messages.append(Msg('assistant', parts))
    elif t=='user' and isinstance(c, list) and c and all(b.get('type')=='tool_result' for b in c):
        self.messages.append(Msg('tool', [ToolResult(id=b.get('tool_use_id'), name=self._names.get(b.get('tool_use_id')),
            text=_flat(b.get('content',''))) for b in c]))
    elif t=='result': self.result = m

`_kick` performs the control handshake and sends the live user turn. The drive starts it as a task because the same protocol event loop must already be consuming the handshake response.

In [ ]:
#| export
@patch
async def _kick(self:ClaudeRun, prompt):
    "Handshake, then send the live user turn"
    await self.proto.initialize()
    await self.proto.send(dict(type='user', message=dict(role='user', content=prompt)))

`_event_type` separates Anthropic's nested SSE event names from Claude CLI's top-level messages. Batch boundaries exist only on `stream_event` records; terminal `result` records therefore have no event type.

In [ ]:
#| export
def _event_type(m): return nested_idx(m, 'event', 'type') if m.get('type')=='stream_event' else None

In [ ]:
start_event = dict(type='stream_event', event=dict(type='message_start'))
result_event = dict(type='result', subtype='success')
test_eq(_event_type(start_event), 'message_start')
test_eq(_event_type(result_event), None)
(_event_type(start_event), _event_type(result_event))

A streamed external batch is installed in the broker before its `message_stop` is yielded, so exhausting that iteration leaves a real paused run.

In [ ]:
#| export
@patch
def _fold_event(self:ClaudeRun, m):
    "Track one event and establish a pause at a complete external tool batch"
    et = _event_type(m)
    if et=='message_start': self._batch = []
    self._track(m)
    if et=='message_stop' and self._batch:
        self.proto.broker.begin(self._batch)
        self._pause_ids = [u.id for u in self._batch]
        self.paused = True
        return True
    return False

The drive retains one protocol event iterator across logical turns.

In [ ]:
#| export
@patch
async def _turn(self:ClaudeRun):
    "Stream until the next external tool batch or terminal result"
    if self._closed or self.result is not None: return
    if self.paused: raise RuntimeError('supply tool results with resume() before iterating again')
    first_turn,keep,kick = self.proc is None,False,None
    try:
        if first_turn:
            prompt = await self._spawn()
            kick = asyncio.create_task(self._kick(prompt))
        while True:
            m = await anext(self._events)

            paused = self._fold_event(m)
            yield m
            if paused:
                keep = True
                return
            if m.get('type')=='result': return
    except StopAsyncIteration:
        if self.result is None: raise RuntimeError('Claude stream ended before a result')
    finally:
        if kick:
            kick.cancel()
            with suppress(asyncio.CancelledError): await kick
        if not keep and not self._closed: await self.aclose()

`resume()` validates that every call id is answered, loads the results together, and the next iteration lets Claude collect its serialized MCP replies.

In [ ]:
#| export
@patch
async def resume(self:ClaudeRun, results):
    "Supply the complete paused tool batch; the next iteration continues the same process"
    if not self.paused: raise RuntimeError('Claude is not waiting for tool results')
    results = listify(results)
    ids = [r.id for r in results]
    if len(ids)!=len(set(ids)) or set(ids)!=set(self._pause_ids): raise ValueError(f'tool results must answer exactly {self._pause_ids}')
    for r in results: self.proto.broker.reply(r.id, r.text or '')
    self._pause_ids,self._batch,self.paused = [],[],False

`interrupt` delegates to Claude's native control request, then clears any paused batch because an aborted turn can no longer accept those tool results. The same event iterator remains available to drain the aborted tail.

In [ ]:
#| export
@patch
async def interrupt(self:ClaudeRun, timeout=30):
    "Claude's native interrupt: end the current turn; iterate again to drain the aborted tail"
    res = await self.proto.interrupt(timeout)
    self._pause_ids,self._batch,self.paused = [],[],False
    return res

Cleanup is the part that must work under cancellation, because a host's ctrl-C arrives as exactly that: the consumer task is cancelled, the stream is closed, and this sequence is all that stands between an abandoned turn and an orphaned process.

If the turn is still running, send the native interrupt and drain the tail directly (the consumer's read loop is gone, so `aclose` reads for itself, still folding what arrives into the trace).

In [ ]:
#| export
@patch
async def _drain_tail(self:ClaudeRun):
    "Fold the aborted process tail into this run"
    async for m in read_msgs(self.proc.stdout):
        if m.get('type')=='control_response': self.proto._resolve(m)
        else: self._track(m)
        if m.get('type')=='result': return

In [ ]:
#| export
@patch
async def _drain(self:ClaudeRun, timeout=10):
    "Bound tail draining so cleanup cannot wait forever"
    with suppress(Exception): await asyncio.wait_for(self._drain_tail(), timeout)

`_abort` coordinates the two halves: start the bounded tail reader first, send Claude's native interrupt, then wait until the terminal tail has been folded into the run.

In [ ]:
#| export
@patch
async def _abort(self:ClaudeRun, proc):
    "Interrupt a live turn and drain its terminal tail"
    if self.result is None and proc.stdin and not proc.stdin.is_closing():
        t = asyncio.create_task(self._drain())
        with suppress(Exception): await asyncio.wait_for(self.proto.interrupt(), 5)
        await t

Then close stdin, wait, and escalate: terminate, then kill, each with a deadline.

`_wait_process` turns a bounded process wait into a boolean, so escalation can be expressed as a flat sequence rather than nested timeout handlers.

In [ ]:
#| export
async def _wait_process(proc, timeout=5):
    try:
        await asyncio.wait_for(proc.wait(), timeout)
        return True
    except (TimeoutError, asyncio.TimeoutError): return False

`_reap_process` first offers a normal wait, then termination, then killing. Each successful stage returns immediately; only the final kill is unbounded because the process must not survive cleanup.

In [ ]:
#| export
async def _reap_process(proc):
    if await _wait_process(proc): return
    with suppress(ProcessLookupError): proc.terminate()
    if await _wait_process(proc): return
    with suppress(ProcessLookupError): proc.kill()
    with suppress(Exception): await proc.wait()

Finally remove the exact transcript this run wrote, success or failure.

In [ ]:
#| export
@patch
async def _cleanup(self:ClaudeRun):
    p = self.proc
    if p and p.returncode is None:

        await self._abort(p)
        with suppress(Exception): p.stdin.close()
        await _reap_process(p)

    if self._events:
        with suppress(Exception): await self._events.aclose()
    if self.proto: await self.proto.aclose()
    if self._spath: Path(self._spath).unlink(missing_ok=True)

The whole body runs shielded, so a second cancellation cannot abort it partway:

In [ ]:
#| export
@patch
async def aclose(self:ClaudeRun):
    "Interrupt if mid-turn, drain, close, escalate, and remove the transcript; idempotent and cancellation-shielded"
    if self._closed: return
    self._closed = True
    await asyncio.shield(asyncio.create_task(self._cleanup()))

## A scripted run

The lifecycle runs end to end against a scripted stand-in for `claude`, so the run's own logic - kickoff ordering, trace accumulation, terminal handling, transcript cleanup - is verified offline. The script answers any control request, echoes the user turn as an assistant message, and finishes with a result:

In [ ]:
# chkstyle: ignore-node
fake_src = textwrap.dedent('''
    #!/usr/bin/env python3
    import sys, json
    def w(o): sys.stdout.write(json.dumps(o)+'\\n'); sys.stdout.flush()
    for line in sys.stdin:
        m = json.loads(line)
        if m.get('type')=='control_request':
            w(dict(type='control_response', response=dict(subtype='success', request_id=m['request_id'], response={})))
        elif m.get('type')=='user':
            c = m['message']['content']
            txt = 'echo: '+(c if isinstance(c, str) else c[0].get('text',''))
            w(dict(type='assistant', message=dict(role='assistant', content=[dict(type='text', text=txt)])))
            w(dict(type='result', subtype='success', result=txt))
    ''').strip()

Writing that peer as an executable preserves the real subprocess, pipe, handshake, and cleanup boundaries while keeping the lesson deterministic and model-free.

In [ ]:
fake_cc = Path(tempfile.mkdtemp())/'claude'
fake_cc.write_text(fake_src+'\n')
fake_cc.chmod(fake_cc.stat().st_mode | stat.S_IXUSR)

In [ ]:
scratch = Path(tempfile.mkdtemp())
run = astream(h, claude_path=fake_cc, cwd=scratch)
got = [m async for m in run]
test_eq(run.result['result'], 'echo: And in gauss?')
test_eq([m.role for m in run.messages], ['assistant'])
test_eq(run.messages[0].content[0].text, 'echo: And in gauss?')
test_eq(run._spath.exists(), False)
[m['type'] for m in got]

## Live runs

The examples below use the real authenticated CLI (they spend tokens, so stay out of automated runs). First Claude chooses the advertised tool and reaches a natural MCP pause. The complete ToolUse batch is present in the trace, `.paused` is true, and the process remains alive. `flux_meter` contributes only its schema: its Python body has not run and the MCP call still has no result:


In [ ]:
async def flux_meter(unit:str='kf') -> str:
    "Read the flux."
    return f'flux: 41.7 {unit}'

[('assistant', ['Thinking']), ('assistant', ['ToolUse'])]

`flux_meter` is advertised as a schema but never invoked by FastClaude. Iterating the run consumes one logical Claude turn and stops at the external tool boundary.

In [ ]:
#| eval: false
lmsgs = [Msg('user', [Text('Use the flux_meter tool with unit="gauss", then reply with exactly the tool output.')])]
lrun = astream(lmsgs, model='claude-sonnet-5', tools=[flux_meter])
levs = [m async for m in lrun]
L(m.get('type') for m in levs).unique()

The accumulated trace ends with Claude's unqualified `ToolUse`. The process remains alive and its pid is retained for the continuation, but no result has yet been supplied.

In [ ]:
#| eval: false
ltu = lrun.messages[-1].content[-1]
test_eq((type(ltu), ltu.name, ltu.arguments), (ToolUse, 'flux_meter', dict(unit='gauss')))
assert lrun.paused
lpid = lrun.proc.pid
[(m.role, [type(p).__name__ for p in m.content]) for m in lrun.messages]

The caller executes the requested batch and resumes this run with matching `ToolResult`s. Claude collects them through its still-live MCP requests and continues in the same process; the resulting tool and assistant messages join the canonical trace normally:


In [ ]:
#| eval: false
out = await flux_meter(**ltu.arguments)
await lrun.resume([ToolResult(id=ltu.id, name=ltu.name, text=out)])
cevs = [m async for m in lrun]
test_eq(lrun.proc.pid, lpid)
test_eq([m.role for m in lrun.messages[-2:]], ['tool','assistant'])
lrun.result['result']


'flux: 41.7 gauss'

Interruption still matters mid-generation: `run.interrupt()` sends the native control request, the stream stays open to drain the aborted tail, and the terminal result reports the interruption distinctly from a provider failure. Mid-tool interruption is no longer this library's concern, because tools run in the caller, who cancels its own work:

In [ ]:
#| eval: false
irun = astream([Msg('user', [Text('Write a 2000 word essay on the history of magnetometry. Do not stop early.')])])
n = 0
async for m in irun:
    if m.get('type')=='stream_event' and (n := n+1)==20: asyncio.ensure_future(irun.interrupt())
test_eq([m.role for m in irun.messages], ['assistant'])
{k: irun.result.get(k) for k in ('subtype','is_error')}


{'subtype': 'error_during_execution', 'is_error': True}

Statelessness between user turns closes the loop: the finished exchange replays as ordinary history under a new prompt. Signed thinking and the completed tool exchange remain available for reference, but nothing re-executes:


In [ ]:
#| eval: false
qmsgs = lmsgs + lrun.messages + [Msg('user', [Text('What unit did the flux_meter report in? One word only.')])]
qrun = astream(qmsgs, tools=[flux_meter])
qevs = [m async for m in qrun]
qrun.result['result']


'Gauss'

## Cleanup

Runs remove their own transcripts, so only empty per-run project folders remain; the scripted run's scratch folder is ours to delete, and the shared cache-dir folder is scratch by contract.

In [ ]:
shutil.rmtree(sess_dir(scratch), ignore_errors=True)
shutil.rmtree(sess_dir(work_dir()), ignore_errors=True)
shutil.rmtree(fake_cc.parent, ignore_errors=True)
shutil.rmtree(scratch, ignore_errors=True)

In [ ]:
#| hide
#| eval: false
import nbdev; nbdev.nbdev_export()